# 11. Full data: applying the feature sets to Paoay

## Setup

In [ ]:
from pathlib import Path
import json
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").is_file():
    if root == root.parent:
        raise RuntimeError("Open this notebook inside the project repository")
    root = root.parent
if str(root / "src") not in sys.path:
    sys.path.insert(0, str(root / "src"))
if str(root) not in sys.path:
    sys.path.insert(0, str(root))  # experiments package

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from experiments.pilots import climate, clustering, data, earth_engine, evaluation, features, full
from experiments.tuning import tuning
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
AREA = "pilots"


def show_and_save(figure, run, name):
    figure.tight_layout()
    figure.savefig(run / f"{name}.png", dpi=150)
    display(figure)
    plt.close(figure)


def repeated(name, X, rows, ok, seeds=(1, 2, 3), params=evaluation.PARAMS):
    """Mean and SD over random-forest seeds of every evaluation view."""
    summaries, details = [], []
    for seed in seeds:
        summary, holdout = evaluation.evaluate(name, X, rows, ok, params, seed)
        summaries.append({**summary, "seed": seed})
        details.extend({**r, "seed": seed} for r in holdout)
    return summaries, details


def summarize(summaries):
    table = pd.DataFrame(summaries)
    metrics = ["random_split_macro_f1", "spatial_cv_macro_f1", "dataset_holdout_macro_f1", "jecam_macro_f1",
               "jecam_carrot_f1", "jecam_cabbage_f1", "jecam_potato_f1"]
    grouped = table.groupby("feature_set", sort=False)
    out = grouped[metrics].agg(["mean", "std"]).round(3)
    out.insert(0, ("pixels_used", ""), grouped["pixels_used"].first())
    return out

## Inputs

In [ ]:
full_run = ""  # Optional pipeline_outputs/pilots/subset_<id>; blank uses the latest
full_run = Path(full_run) if full_run else tuning.latest_run(root, "full_data", area=AREA)
rows, arrays = data.load(full_run)
cube = features.stack(arrays["s2_values"])
valid = arrays["s2_valid"]
print("Full data:", full_run, "| pixels:", len(rows))

In [ ]:
evaluation_run = ""  # Optional pipeline_outputs/pilots/full_evaluation_<id>; blank uses the latest
evaluation_run = Path(evaluation_run) if evaluation_run else tuning.latest_run(root, "full_evaluation", area=AREA)
selected = json.loads((evaluation_run / "selected_groups.json").read_text())
paoay = json.loads((full_run / "paoay.json").read_text())
pixels, p_s2, p_valid, p_s1 = full.paoay_pixels(arrays)
s2, s1 = arrays["s2_values"], arrays["s1_values"]
statistics = pd.read_csv(root / "data/paoay/benguet_vegetable_area_2024.csv")
benguet = dict(zip(statistics.crop, statistics.area_ha / statistics.area_ha.sum()))
print("Paoay pixels:", len(pixels), "| evaluation run:", evaluation_run, "| selected groups:", selected)

## Cropland mask

In [ ]:
import rasterio
from rasterio.warp import transform as reproject_points
from crop_pipeline.analysis_workflow import resolve_stage_pointer
handoff = resolve_stage_pointer(root / "sen4agrinet_work/paoay/land_activity/land_activity_stage.json")
grid = paoay["grid"]
centres = [rasterio.Affine(*grid["transform"]) * (c + .5, r + .5) for r, c in pixels]
with rasterio.open(handoff.parent / "land_activity.tif") as raster:
    xs, ys = reproject_points(grid["crs"], raster.crs, [x for x, _ in centres], [y for _, y in centres])
    cropland = np.asarray([value[0] for value in raster.sample(zip(xs, ys))]) == 1
print("Active cropland pixels:", int(cropland.sum()), "of", len(pixels))

## Train on all development parcels and apply

In [ ]:
run = tuning.new_run(root, "full_paoay", full_run / "meta.json", area=AREA, evaluation_run=str(evaluation_run))
development = np.asarray([r["partition"] in ("train", "validation") for r in rows])
rule = {"minimum_top_probability": .6, "minimum_margin": .1}
configurations = {name: (full.SETS[name], False) for name in ("s2_harmonic", "s1_ratios", "s2_harmonic_plus_s1_ratios")}
configurations["s2_harmonic + augmentation"] = (full.SETS["s2_harmonic"], True)
configurations["s2_harmonic_plus_s1_ratios + augmentation"] = (full.SETS["s2_harmonic_plus_s1_ratios"], True)
configurations["selected_groups"] = (full.SETS["s2_harmonic_plus_s1_raw_and_ratios"], False)
labels, records = {}, []
for name, (build, augment) in configurations.items():
    X, ok, columns = build(s2, valid, s1)
    pX, pok, _ = build(p_s2, p_valid, p_s1)
    keep = list(range(X.shape[1]))
    if name == "selected_groups":
        groups = features.column_groups(columns)
        keep = [c for g in selected for c in groups[g]]
    index = np.nonzero(development & ok)[0]
    copies = [c[:, keep] for c in full.copies(build, s2, valid, s1)] if augment else None
    model = evaluation.fit_rows(np.nan_to_num(X[:, keep]), rows, index, {**evaluation.PARAMS, "n_estimators": 500}, copies=copies)
    probabilities = model.predict_proba(np.nan_to_num(pX[:, keep]))
    ordered = np.sort(probabilities, axis=1)
    accepted = pok & (ordered[:, -1] >= rule["minimum_top_probability"]) & (ordered[:, -1] - ordered[:, -2] >= rule["minimum_margin"])
    label = np.where(~pok, "No features", np.where(accepted, model.classes_[np.argmax(probabilities, axis=1)], "Unknown crop type"))
    labels[name] = label
    crop_labels = pd.Series(label[cropland & accepted])
    shares = crop_labels.value_counts(normalize=True)
    records.append({"configuration": name, "pixels_with_features": float(pok.mean()),
                    "confident_share_in_cropland": float(accepted[cropland].mean()),
                    **{f"{c}_share_of_confident_cropland": float(shares.get(c, 0)) for c in evaluation.CROPS}})
    print(name, "applied", flush=True)
table = pd.DataFrame(records)
table.to_csv(run / "paoay_shares.csv", index=False)
display(table.round(3))
print("Benguet 2024 shares:", {k: round(v, 3) for k, v in benguet.items()})

## Reference areas

In [ ]:
from crop_pipeline.paoay import aligned
from rasterio.features import geometry_mask
from shapely.geometry import mapping
mask = arrays["paoay_mask"]
settings = json.loads((root / "settings.local.json").read_text())
_, areas, _ = aligned.study_area([Path(p) for p in paoay["kmz_paths"]], settings.get("paoay_roi_name", "Region of Interest"),
                                 list(settings.get("paoay_context_names", [])), grid["crs"])
area_rows = []
for name, label in labels.items():
    raster = np.full(mask.shape, "", dtype=object)
    raster[tuple(pixels.T)] = label
    for area in areas:
        inside = ~geometry_mask([mapping(area["geometry"])], out_shape=mask.shape, transform=rasterio.Affine(*grid["transform"])) & mask
        area_rows.append({"configuration": name, "area": area["name"], **pd.Series(raster[inside]).value_counts().to_dict()})
area_table = pd.DataFrame(area_rows).fillna(0)
area_table.to_csv(run / "reference_areas.csv", index=False)
display(area_table)

## Maps

In [ ]:
codes = {"carrot": 1, "cabbage": 2, "potato": 3, "Unknown crop type": 4, "No features": 5}
figure, axes = plt.subplots(2, 3, figsize=(15, 10))
for axis, (name, label) in zip(axes.ravel(), labels.items()):
    image = np.full(mask.shape, np.nan)
    image[tuple(pixels.T)] = [codes[l] for l in label]
    axis.imshow(image, cmap="Set2", vmin=1, vmax=8, interpolation="nearest")
    axis.set_title(name, fontsize=9)
    axis.axis("off")
show_and_save(figure, run, "paoay_maps")
print("Run folder:", run)